In [1]:
import sys
from pathlib import Path

import os

# Set the working directory to the root of the project
os.chdir(Path('..').resolve())  # Adjust this path as necessary

# Now append the src directory to the path
import sys
sys.path.append(str(Path('src').resolve()))

import numpy as np
import pandas as pd
# Now import the necessary modules
from config import Config
import argparse
import logging
from pathlib import Path
from config import Config, setup_logging
from model import get_models
from data_loader import (
    StockDataset,
    download_stock_data,
    get_sp500_tickers,
    prepare_data,
    create_dataloader
)
from trainer import GANTrainer
from evaluation import (
    generate_synthetic_data,
    evaluate_quality,
    plot_results,
    plot_training_history,
    analyze_distributions
)
# Initialize configuration and check the data directory
config = Config()
print(f"Data directory: {config.DATA_DIR}")

import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


# stock_data
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import pywt  # For wavelet transformations
import random
import logging
from typing import List, Dict
import matplotlib.pyplot as plt
import torch.nn.functional as F

# Setting up logger
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("GAN_Training")


import torch
import torch.nn as nn
import math
import torch

class DifferentialPrivacy:
    """
    A class to apply differential privacy mechanisms to gradients by adding noise.

    Args:
        epsilon (float): Privacy parameter to control the amount of noise added. Lower values increase privacy.
        delta (float): Privacy parameter that controls the probability of preserving privacy. Lower values increase privacy.
    """
    def __init__(self, epsilon: float, delta: float):
        self.epsilon = epsilon
        self.delta = delta
        self.sensitivity = 1.0

    def apply_gradient_noise(self, gradients: torch.Tensor) -> torch.Tensor:
        """
        Apply Laplace noise to gradients to ensure differential privacy.

        Args:
            gradients (torch.Tensor): Gradients of the model parameters.

        Returns:
            torch.Tensor: Gradients with added noise for differential privacy.
        """
        noise = torch.distributions.laplace.Laplace(0, self.sensitivity / self.epsilon).sample(gradients.size()).to(gradients.device)
        return gradients + noise


class PositionalEncoding(nn.Module):
    """
    Adds positional encoding to input embeddings to retain positional information in the model.

    Args:
        hidden_size (int): Dimensionality of the embeddings.
        max_len (int): The maximum length of sequences for which to compute positional encodings.
    """
    def __init__(self, hidden_size: int, max_len: int = 5000):
        super(PositionalEncoding, self).__init__()
        encoding = torch.zeros(max_len, hidden_size)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, hidden_size, 2).float() * (-math.log(10000.0) / hidden_size))
        encoding[:, 0::2] = torch.sin(position * div_term)
        encoding[:, 1::2] = torch.cos(position * div_term)
        encoding = encoding.unsqueeze(0)
        self.register_buffer('positional_encoding_buffer', encoding)

    def forward(self, x):
        """
        Forward pass to add positional encoding to the input tensor.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, sequence_length, hidden_size).

        Returns:
            torch.Tensor: Output tensor with positional encoding added.
        """
        return x + self.positional_encoding_buffer[:, :x.size(1), :]


# Step 1: Custom Dataset for Stock Data with Date Information
class StockDataset(Dataset):
    def __init__(self, data: Dict[str, pd.DataFrame]):
        """
        Custom dataset for stock data with date encoding.

        Args:
            data (Dict[str, pd.DataFrame]): Dictionary where keys are stock tickers and values are
                                             historical data DataFrames with 'Open', 'High', 'Low', 
                                             'Close', 'Volume' columns.
        """
        self.data = []
        for ticker, df in data.items():
            features = df[['Open', 'High', 'Low', 'Close', 'Volume']].values
            self.data.extend(features)
        self.data = torch.tensor(self.data, dtype=torch.float32).to(Config().DEVICE)

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int) -> torch.Tensor:
        return self.data[idx]

# Step 2: Define a Wavelet Loss Function
class WaveletLoss(nn.Module):
    def __init__(self, wavelet: str = "haar"):
        """
        Wavelet-based loss function for measuring frequency similarity between real and synthetic data.

        Args:
            wavelet (str): The type of wavelet to use (e.g., 'haar', 'db1', etc.).
        """
        super(WaveletLoss, self).__init__()
        self.wavelet = wavelet

    def forward(self, real: torch.Tensor, synthetic: torch.Tensor) -> torch.Tensor:
        """
        Compute wavelet loss between real and synthetic data, with reshaping to ensure 3D alignment if necessary.

        Args:
            real (torch.Tensor): Real data tensor of shape (batch_size, sequence_length, feature_size) or (batch_size, sequence_length).
            synthetic (torch.Tensor): Synthetic data tensor with the same shape as `real`.

        Returns:
            torch.Tensor: The computed wavelet-based loss.
        """
        # Ensure both real and synthetic are 3D: (batch_size, sequence_length, feature_size)
        if real.dim() == 2:  # If 2D, add a feature dimension of size 1
            real = real.unsqueeze(-1)
        if synthetic.dim() == 2:
            synthetic = synthetic.unsqueeze(-1)

        # Ensure both tensors have the same sequence length
        min_dim = min(real.size(1), synthetic.size(1))
        real = real[:, :min_dim, :]
        synthetic = synthetic[:, :min_dim, :]

        # Compute wavelet coefficients
        real_coeffs = pywt.wavedec(real.cpu().numpy(), self.wavelet, axis=1)
        synthetic_coeffs = pywt.wavedec(synthetic.cpu().detach().numpy(), self.wavelet, axis=1)

        # Calculate MSE loss on each pair of coefficients
        loss = sum(
            F.mse_loss(torch.tensor(r, dtype=torch.float32), torch.tensor(s, dtype=torch.float32))
            for r, s in zip(real_coeffs, synthetic_coeffs)
        )
        return loss.to(real.device)




# Step 3: WGAN Generator with Transformer and Date Embedding
class WGANGenerator(nn.Module):
    def __init__(self, config):
        """
        Generator network using Transformer layers for sequence modeling.
        
        Args:
            config (Config): Configuration object with model parameters.
        """
        super(WGANGenerator, self).__init__()
        self.noise_dim = config.NOISE_DIM
        self.hidden_size = config.HIDDEN_SIZE
        self.sequence_length = config.SEQUENCE_LENGTH
        self.embedding_dim = config.EMBEDDING_DIM
        self.num_stocks = 3
        self.feature_size = config.FEATURE_SIZE
        self.n_heads = config.N_HEADS
        self.n_layers = config.N_LAYERS
        self.dropout = config.DROPOUT

        # Stock embedding and noise embedding
        self.stock_embedding = nn.Embedding(self.num_stocks, self.embedding_dim)
        self.noise_embedding = nn.Linear(self.noise_dim, self.hidden_size - self.embedding_dim)
        
        # Positional encoding for sequential data
        self.positional_encoding = PositionalEncoding(self.hidden_size)
        
        # Transformer blocks
        self.transformer_blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(d_model=self.hidden_size, nhead=self.n_heads, dropout=self.dropout)
            for _ in range(self.n_layers)
        ])
        
        # Output layer to produce final feature dimensions
        self.output_layer = nn.Sequential(
            nn.Linear(self.hidden_size, self.hidden_size),
            nn.LayerNorm(self.hidden_size),
            nn.GELU(),
            nn.Linear(self.hidden_size, self.feature_size)
        )

    def forward(self, noise: torch.Tensor, stock_id: torch.Tensor) -> torch.Tensor:
        """
        Forward pass for the generator.

        Args:
            noise (torch.Tensor): Input noise tensor of shape (batch_size, noise_dim).
            stock_id (torch.Tensor): Tensor containing stock IDs for embedding lookup.

        Returns:
            torch.Tensor: Generated synthetic data with shape (batch_size, sequence_length, feature_size).
        """
        # Stock embedding and noise projection
        stock_embed = self.stock_embedding(stock_id).unsqueeze(1).repeat(1, self.sequence_length, 1)
        noise_proj = self.noise_embedding(noise).unsqueeze(1).repeat(1, self.sequence_length, 1)
        
        # Concatenate the expanded noise projection and stock embedding
        x = torch.cat([noise_proj, stock_embed], dim=-1)
        x = self.positional_encoding(x)

        # Apply Transformer blocks
        for block in self.transformer_blocks:
            x = block(x)

        return self.output_layer(x)



# Step 4: WGAN Discriminator with Regularization
class WGANDiscriminator(nn.Module):
    def __init__(self, config: Config):
        """
        Initialize the WGAN Discriminator with regularization.

        Args:
            config (Config): Configuration object with model parameters.
        """
        super(WGANDiscriminator, self).__init__()
        self.fc1 = nn.Linear(config.FEATURE_SIZE, config.HIDDEN_SIZE)
        self.fc2 = nn.Linear(config.HIDDEN_SIZE, 1)
        
        # Apply spectral normalization for stability
        self.apply_spectral_norm(self.fc1)
        self.apply_spectral_norm(self.fc2)
    
    @staticmethod
    def apply_spectral_norm(layer):
        nn.utils.spectral_norm(layer)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass of the Discriminator.

        Args:
            x (torch.Tensor): Input data tensor of shape (batch_size, feature_size).

        Returns:
            torch.Tensor: Discriminator output, probability of real or fake.
        """
        x = torch.relu(self.fc1(x))
        return torch.sigmoid(self.fc2(x))

# Step 5: Training WGAN with Differential Privacy and Wavelet Loss
def train_wgan_with_wavelet_loss(generator, discriminator, data_loader, dp_mechanism, wavelet_loss, config: Config, epochs=10):
    """
    Train the WGAN with differential privacy by adding noise to gradients and applying wavelet loss.

    Args:
        generator (WGANGenerator): Generator model.
        discriminator (WGANDiscriminator): Discriminator model.
        data_loader (DataLoader): DataLoader for real data.
        dp_mechanism (DifferentialPrivacy): Differential privacy mechanism for gradient noise.
        wavelet_loss (WaveletLoss): Wavelet loss function.
        config (Config): Configuration object with training parameters.
        epochs (int): Number of training epochs.
    """
    config.NUM_STOCKS = 3
    optimizer_g = optim.Adam(generator.parameters(), lr=config.GEN_LR, betas=(config.BETA1, config.BETA2), weight_decay=config.WEIGHT_DECAY)
    optimizer_d = optim.Adam(discriminator.parameters(), lr=config.DISC_LR, betas=(config.BETA1, config.BETA2), weight_decay=config.WEIGHT_DECAY)
    
    for epoch in range(epochs):
        for batch in data_loader:
            batch = batch.to(config.DEVICE)
            
            # Discriminator training
            optimizer_d.zero_grad()
            real_labels = discriminator(batch)
            noise = torch.randn(batch.size(0), config.NOISE_DIM).to(config.DEVICE)
            fake_data = generator(noise, torch.randint(0, config.NUM_STOCKS, (batch.size(0),)).to(config.DEVICE))
            fake_labels = discriminator(fake_data.detach())
            
            d_loss = -(torch.mean(real_labels) - torch.mean(fake_labels))
            d_loss.backward()
            
            # Apply gradient noise for differential privacy
            for param in discriminator.parameters():
                param.grad = dp_mechanism.apply_gradient_noise(param.grad)
            
            optimizer_d.step()
            
            # Generator training
            optimizer_g.zero_grad()
            fake_labels = discriminator(fake_data)
            g_loss = -torch.mean(fake_labels)
            
            # Wavelet loss addition
            wavelet_penalty = wavelet_loss(batch, fake_data)
            total_loss = g_loss + 0.1 * wavelet_penalty  # Adjust weight if needed
            total_loss.backward()
            
            # Apply gradient noise for differential privacy
            for param in generator.parameters():
                param.grad = dp_mechanism.apply_gradient_noise(param.grad)
            
            optimizer_g.step()
        
        logger.info(f"Epoch [{epoch+1}/{epochs}], D Loss: {d_loss.item()}, G Loss: {g_loss.item()}, Wavelet Loss: {wavelet_penalty.item()}")

# Step 6: Evaluation and Visualization
def evaluate_and_plot(generator, real_data: pd.DataFrame, num_samples: int = 100):
    """
    Generate synthetic data using the trained generator and evaluate quality by comparing with real data.

    Args:
        generator (WGANGenerator): Trained generator model.
        real_data (pd.DataFrame): Real stock data for comparison.
        num_samples (int): Number of samples to generate for comparison.
    """
    generator.eval()
    config = Config()
    config.NUM_STOCKS = 3
    with torch.no_grad():
        noise = torch.randn(num_samples, config.NOISE_DIM).to(config.DEVICE)
        synthetic_data = generator(noise, torch.randint(0, config.NUM_STOCKS, (num_samples,)).to(config.DEVICE)).cpu().numpy()
    
    # Convert real and synthetic data into a comparable format
    real_data_sample = real_data.sample(n=num_samples).reset_index(drop=True)
    synthetic_data_df = pd.DataFrame(synthetic_data, columns=real_data_sample.columns)
    
    # Plot real vs synthetic distributions for each feature
    plt.figure(figsize=(12, 10))
    for i, column in enumerate(real_data_sample.columns):
        plt.subplot(3, 2, i + 1)
        plt.hist(real_data_sample[column], bins=30, alpha=0.6, color='blue', label='Real')
        plt.hist(synthetic_data_df[column], bins=30, alpha=0.6, color='orange', label='Synthetic')
        plt.title(f"Distribution of {column}")
        plt.legend()

    plt.tight_layout()
    plt.show()
    
    # Time-series comparison of real and synthetic data for a single sample sequence
    sample_index = np.random.randint(0, num_samples)
    plt.figure(figsize=(14, 8))
    for i, column in enumerate(real_data_sample.columns):
        plt.subplot(3, 2, i + 1)
        plt.plot(real_data_sample.index, real_data_sample[column], color='blue', label='Real')
        plt.plot(synthetic_data_df.index, synthetic_data_df[column], color='orange', linestyle='--', label='Synthetic')
        plt.title(f"{column} Over Time")
        plt.legend()

    plt.tight_layout()
    plt.show()



Data directory: data


In [2]:
# Fetch S&P 500 tickers and download stock data
tickers = get_sp500_tickers(config)
stock_data = download_stock_data(tickers, config)

INFO:data_loader:Loaded data for 495 stocks


data/stock_data.pkl


In [3]:
num_stocks = 3
rows_per_stock = 700

np.random.seed(31)

# Select stocks if num_stocks is specified
if num_stocks is not None:
    selected_stocks = np.random.choice(list(stock_data.keys()), num_stocks, replace=False)
else:
    selected_stocks = list(stock_data.keys())
print("Selected stocks:", selected_stocks)

# Create a mapping of stock names to unique IDs
stock_to_id = {stock: idx for idx, stock in enumerate(selected_stocks)}
print('stock_to_id:', stock_to_id)

# Get the full date range from the stock data
min_date = min(df.index.min() for df in stock_data.values())
max_date = max(df.index.max() for df in stock_data.values())
all_dates = pd.date_range(start=min_date, end=max_date, freq='B')

# Initialize dictionaries
data_dict = {}
aligned_data_dict = {}
mask_dict = {}
normalization_params = {}

for stock in selected_stocks:
    # Align stock data with the full date range
    df = stock_data[stock]
    df_aligned = df.reindex(all_dates)
    mask = df_aligned.notna().astype(int)
    
    data_dict[stock] = df_aligned.iloc[:rows_per_stock]
    # print('df_aligned')
    # display(df_aligned)
    
    # Normalize data
    # normalized_data = robust_normalize(df_aligned.values)
    normalized_df = pd.DataFrame(df_aligned, index=all_dates, columns=df.columns)
    # print('normalized_df')
    # display(normalized_df)
    normalized_df[mask == 0] = np.nan  # Keep NaNs where the original data is missing

    # Now, select only the specified number of non-NaN rows
    non_nan_rows = normalized_df.dropna().iloc[:rows_per_stock]

    # Store only the selected rows in the final dictionaries
    aligned_data_dict[stock] = non_nan_rows
    mask_dict[stock] = mask.loc[non_nan_rows.index]
    normalization_params[stock] = (
        np.nanmedian(df_aligned.values, axis=0),
        np.nanpercentile(df_aligned.values, 75, axis=0) - np.nanpercentile(df_aligned.values, 25, axis=0)
    )

Selected stocks: ['QCOM' 'BRO' 'ACN']
stock_to_id: {'QCOM': 0, 'BRO': 1, 'ACN': 2}


In [4]:
data_dict

{'QCOM':                   Open        High         Low       Close     Volume
 2020-01-02   89.050003   89.809998   88.080002   88.690002  8413900.0
 2020-01-03   87.260002   87.639999   86.440002   87.019997  8340300.0
 2020-01-06   85.910004   86.550003   85.540001   86.510002  8381400.0
 2020-01-07   87.040001   89.489998   86.910004   88.970001  8377400.0
 2020-01-08   88.900002   89.470001   87.919998   88.709999  7619900.0
 ...                ...         ...         ...         ...        ...
 2022-09-01  129.979996  130.130005  126.080002  129.919998  8716300.0
 2022-09-02  131.639999  132.669998  127.559998  128.479996  6121400.0
 2022-09-05         NaN         NaN         NaN         NaN        NaN
 2022-09-06  128.839996  129.520004  126.209999  126.669998  6674200.0
 2022-09-07  127.470001  129.779999  126.370003  128.600006  5340100.0
 
 [700 rows x 5 columns],
 'BRO':                  Open       High        Low      Close     Volume
 2020-01-02  39.619999  39.669998  39.2

In [5]:
config = Config()
dataset = StockDataset(data_dict)
data_loader = DataLoader(dataset, batch_size=config.BATCH_SIZE, shuffle=True)

generator = WGANGenerator(config).to(config.DEVICE)
discriminator = WGANDiscriminator(config).to(config.DEVICE)
dp_mechanism = DifferentialPrivacy(epsilon=1.0, delta=1e-5)
wavelet_loss = WaveletLoss()

train_wgan_with_wavelet_loss(generator, discriminator, data_loader, dp_mechanism, wavelet_loss, config, epochs=10)
evaluate_and_plot(generator, data_dict['QCOM'], num_samples=100)


/var/folders/pk/p4tz0gf96l113931p5pw8d6c0000gp/T/ipykernel_81209/1007145014.py:144: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:278.)
  self.data = torch.tensor(self.data, dtype=torch.float32).to(Config().DEVICE)
/var/folders/pk/p4tz0gf96l113931p5pw8d6c0000gp/T/ipykernel_81209/1007145014.py:192: UserWarning: Using a target size (torch.Size([64, 2, 5])) that is different to the input size (torch.Size([64, 2, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  F.mse_loss(torch.tensor(r, dtype=torch.float32), torch.tensor(s, dtype=torch.float32))
/var/folders/pk/p4tz0gf96l113931p5pw8d6c0000gp/T/ipykernel_81209/1007145014.py:192: UserWarning: Using a target size (torch.Size([64, 3, 5])) t

KeyboardInterrupt: 